# 03b · Full-frame 회귀와 승자 재학습

이 파일은 DCC 2026 작업 노트북의 **공개용 보존본**입니다. 아래 원래 셀과 텍스트 출력은 저장된 실험 기록이며, 이번 공개 작업에서 모델 학습·추론·평가를 다시 실행하지 않았습니다.

- **목적:** 같은 shard에서 backbone을 비교하거나, pilot 선택에서 동결한 backbone을 full 데이터로 재학습합니다.
- **입력:** 03A dataset seal과 shard; full_retrain에서는 이전 pilot selection seal
- **출력:** checkpoint, validation 예측·RMSE와 full-frame candidate bundle
- **실제 저장된 실행 상태:** 원래 제목에는 세 가지 비교가 남아 있지만 현재 설정·저장 출력은 full_retrain입니다. ConvNeXt-Tiny를 epoch 3에서 재개해 5epoch까지 진행했고 best epoch=5, validation RMSE=0.26450450986932456 m를 기록합니다. 이 파일의 현재 출력은 ResNet18·EfficientNet-B0·ConvNeXt 세 후보 pilot의 실행 증거가 아닙니다.

**실행 전:** 공개 저장소에는 원천 이미지·라벨 ZIP, private test 연결표·정답, 모델 가중치, 원래 GCS 산출물이 없습니다. `YOUR_GCP_PROJECT_ID`, `YOUR_GCS_BUCKET`과 필요한 데이터·권한을 직접 설정해야 하며, 원래 봉인 파일의 generation·SHA-256·config 검증도 충족해야 합니다. placeholder만 바꾼다고 과거 산출물이 재생성되거나 모든 gate가 통과하는 것은 아닙니다.

**공개본 변경:** 개인 프로젝트·버킷·계정·로컬 데이터 경로를 placeholder로 바꾸고 Colab/Drive/작성자/widget metadata를 제거했습니다. 코드 셀의 순서와 계산·무결성 검증 로직은 보존했습니다. 모든 내장 image/HTML/JavaScript/기타 binary 출력과 markdown 첨부 이미지를 제거하고, 숫자·표·오류를 포함한 텍스트 출력은 식별자를 가린 뒤 보존했습니다. 사진이 없는 것은 해당 셀을 실행하지 않았다는 뜻이 아닙니다. 원본 code cell의 `execution_count`는 모두 `null`이므로 이 값만으로 실행 여부를 판단하지 않습니다.

[GitHub 코드](https://github.com/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/03b_backbone_comparison.ipynb) · [Colab에서 열기](https://colab.research.google.com/github/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/03b_backbone_comparison.ipynb) · [실행 관계와 상태](README.md) · [원본 이름·SHA-256 대응표](../docs/notebook-source-map.csv)

---

아래는 비공개 식별자를 치환한 원래 노트북입니다. 원래 안내의 예정 단계·성공 기준은 위의 실제 저장 상태와 구분해 읽으십시오.


# DCC 2026 Mission 3 — 03B. 세 가지 full-frame 회귀 모델 공정 비교

같은 train shard와 같은 전체 validation 30,808개에서 다음 모델을 비교합니다.

1. ResNet18 + 유형 조건부 회귀
2. EfficientNet-B0 + 유형 조건부 회귀
3. ConvNeXt-Tiny + 유형 조건부 회귀

세 모델은 이미지 tensor와 파일명 첫 문자에서 만든 `type_index`만 입력받습니다. 최종 선택은
detector AP나 학습 loss가 아니라 validation meter RMSE로 수행합니다.


## 런타임과 예상 실행량

GPU 런타임을 사용하십시오. T4 기준 시작값은 224px, batch 64입니다. OOM이면 batch를 32로
줄이십시오. pilot 24,000장 × 3 epoch × 3모델은 GPU 종류에 따라 약 1~4시간 범위로 계획하되,
실제 ETA는 첫 200 batch에서 출력되는 처리율로 판단합니다.

먼저 `RUN_MODE="smoke"`로 데이터·checkpoint·평가 경로를 확인한 뒤
`RUN_MODE="pilot_compare"`를 실행하십시오.
smoke 결과는 Mission 3 성능표에 사용하지 않습니다.


In [ ]:
%pip -q install "google-cloud-storage>=2.19,<4" "pyarrow>=18,<24" "pandas>=2.2,<3" "webdataset>=1.0.2,<2"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.0 MB/s eta 0:00:00


In [ ]:
import gc
import hashlib
import json
import math
import os
import platform
import random
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import webdataset as wds
from google.colab import auth
from PIL import Image
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import models, transforms

PROJECT_ID = "YOUR_GCP_PROJECT_ID"
BUCKET_NAME = "YOUR_GCS_BUCKET"
RUN_MODE = "full_retrain"  # "smoke", "pilot_compare", "full_retrain"
PILOT_SELECTION_ID = "m3-selection-pilot-router-v1"
if RUN_MODE == "full_retrain":
    DATASET_VERSION = "m3ds-v2-pilot"
    DATA_PROFILE = "full"
    RUN_ID = "m3-fullframe-router-full-v2"
    RUN_LEVEL = "full"
    CANDIDATES = []  # pilot seal의 동결 architecture로 인증 후 채웁니다.
else:
    DATASET_VERSION = "m3ds-v1-pilot"
    DATA_PROFILE = "pilot"
    RUN_ID = "m3-fullframe-smoke-v1" if RUN_MODE == "smoke" else "m3-fullframe-pilot-v1"
    RUN_LEVEL = "smoke" if RUN_MODE == "smoke" else "pilot"
    CANDIDATES = ["resnet18", "efficientnet_b0", "convnext_tiny"]
IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2
EPOCHS = 5 if RUN_MODE == "full_retrain" else (1 if RUN_MODE == "smoke" else 3)
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
AUGMENT = False
SEED = 42
MAX_TRAIN_BATCHES = 2 if RUN_LEVEL == "smoke" else None
MAX_VAL_BATCHES = 2 if RUN_LEVEL == "smoke" else None
EXPECTED_FULL_VAL_N = 30_808

LOCAL_ROOT = Path("/content/dcc_mission3_03b")
# 같은 runtime에서 설정을 바꾸더라도 pilot/full tar 이름이 충돌하지 않도록
# dataset version을 local cache namespace에 포함합니다.
LOCAL_SHARDS = LOCAL_ROOT / "shards" / DATASET_VERSION
LOCAL_RUN = LOCAL_ROOT / "runs" / RUN_ID
LOCAL_META = LOCAL_ROOT / "meta"
for path in [LOCAL_SHARDS, LOCAL_RUN, LOCAL_META]:
    path.mkdir(parents=True, exist_ok=True)

RUN_PREFIX = f"dcc/mission3/runs/{RUN_ID}"
DATASET_SEAL_OBJECT = (
    f"dcc/mission3/datasets/{DATASET_VERSION}/dataset_seal.json"
)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda" and RUN_LEVEL != "smoke":
    raise RuntimeError("pilot/full 학습에는 GPU 런타임이 필요합니다")

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
print("device:", DEVICE)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram GiB:", torch.cuda.get_device_properties(0).total_memory / 1024**3)
print("torch:", torch.__version__, "torchvision:", torchvision.__version__)


device: cuda
gpu: Tesla T4
vram GiB: 14.56317138671875
torch: 2.11.0+cu128 torchvision: 0.26.0+cu128


In [ ]:
from google.api_core.exceptions import NotFound, PreconditionFailed
from google.cloud import storage

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def canonical_sha256(value):
    raw = json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def canonical_target_sha256(frame, target_column):
    """원본 float64와 DataLoader float32가 같은 수위를 같은 byte 계약으로 해시합니다."""
    required = {"stem", "type", target_column}
    missing = required - set(frame.columns)
    if missing:
        raise AssertionError(f"target hash 필수 열 누락: {sorted(missing)}")
    ordered = frame.sort_values("stem", kind="mergesort")
    records = [
        {
            "stem": str(getattr(row, "stem")),
            "type": str(getattr(row, "type")),
            "target_m_float32": float(np.float32(float(getattr(row, target_column)))),
        }
        for row in ordered.itertuples(index=False)
    ]
    return canonical_sha256(records)

def parse_gs_uri(uri):
    if not str(uri).startswith("gs://"):
        raise ValueError(f"GCS URI가 아닙니다: {uri}")
    bucket_name, object_name = str(uri)[5:].split("/", 1)
    return bucket_name, object_name

def download_object(bucket, object_name, local_path, generation=None):
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    blob = bucket.blob(object_name, generation=generation)
    kwargs = {"checksum": "auto"}
    if generation is not None:
        kwargs["if_generation_match"] = int(generation)
    blob.download_to_filename(str(local_path), **kwargs)
    return local_path

def upload_create_only(bucket, local_path, object_name, artifact_type):
    local_path = Path(local_path)
    file_sha = sha256_file(local_path)
    blob = bucket.blob(object_name)
    blob.metadata = {
        "sha256": file_sha,
        "artifact_type": artifact_type,
    }
    try:
        blob.upload_from_filename(
            str(local_path), if_generation_match=0, checksum="auto"
        )
    except PreconditionFailed:
        blob.reload()
        remote_sha = (blob.metadata or {}).get("sha256")
        remote_artifact_type = (blob.metadata or {}).get("artifact_type")
        if (
            remote_sha != file_sha
            or remote_artifact_type != artifact_type
            or int(blob.size or -1) != local_path.stat().st_size
        ):
            raise RuntimeError(
                f"create-only 충돌: gs://{bucket.name}/{object_name}"
            )
    blob.reload()
    if (
        (blob.metadata or {}).get("sha256") != file_sha
        or (blob.metadata or {}).get("artifact_type") != artifact_type
    ):
        raise RuntimeError(f"업로드 SHA metadata 검증 실패: {object_name}")
    return {
        "uri": f"gs://{bucket.name}/{object_name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": file_sha,
        "artifact_type": artifact_type,
    }

def upload_json_create_only(bucket, payload, object_name, artifact_type):
    local_path = LOCAL_META / Path(object_name).name
    local_path.parent.mkdir(parents=True, exist_ok=True)
    local_path.write_text(
        json.dumps(
            payload, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False
        ) + "\n",
        encoding="utf-8",
    )
    return upload_create_only(bucket, local_path, object_name, artifact_type)

def download_json(bucket, object_name, generation=None):
    blob = bucket.blob(object_name, generation=generation)
    kwargs = {}
    if generation is not None:
        kwargs["if_generation_match"] = int(generation)
    return json.loads(blob.download_as_bytes(**kwargs).decode("utf-8"))

def remote_descriptor(bucket, object_name, artifact_type=None):
    blob = bucket.blob(object_name)
    blob.reload()
    metadata = blob.metadata or {}
    file_sha = metadata.get("sha256")
    if not file_sha:
        raise RuntimeError(f"remote object에 sha256 metadata가 없습니다: {object_name}")
    remote_artifact_type = metadata.get("artifact_type")
    if artifact_type is not None and remote_artifact_type != artifact_type:
        raise RuntimeError(
            f"remote artifact_type 불일치: {object_name} "
            f"({remote_artifact_type!r} != {artifact_type!r})"
        )
    return {
        "uri": f"gs://{bucket.name}/{blob.name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": file_sha,
        "artifact_type": remote_artifact_type or "unknown",
    }


In [ ]:
TYPE_ORDER = list("ABCEFGHS")
TYPE_TO_INDEX = {name: index for index, name in enumerate(TYPE_ORDER)}
TYPE_RANGES = {
    "A": (0.0, 15.0), "B": (0.0, 9.0),
    "C": (0.0, 15.0), "E": (0.0, 10.0),
    "F": (0.0, 4.0),  "G": (0.0, 4.0),
    "H": (0.0, 20.0), "S": (0.0, 20.0),
}

class Letterbox:
    def __init__(self, size, fill=(114, 114, 114)):
        self.size = int(size)
        self.fill = fill

    def __call__(self, image):
        image = image.convert("RGB")
        width, height = image.size
        scale = min(self.size / width, self.size / height)
        new_size = (
            max(1, int(round(width * scale))),
            max(1, int(round(height * scale))),
        )
        resized = image.resize(new_size, Image.Resampling.BILINEAR)
        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        offset = ((self.size - new_size[0]) // 2, (self.size - new_size[1]) // 2)
        canvas.paste(resized, offset)
        return canvas

def build_transform(size, training=False, augment=False):
    steps = [transforms.Lambda(Letterbox(size))]
    if training and augment:
        steps.extend([
            transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.06),
            transforms.RandomAffine(
                degrees=2.0, translate=(0.02, 0.02), scale=(0.96, 1.04),
                fill=(114, 114, 114),
            ),
        ])
    steps.extend([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ])
    return transforms.Compose(steps)

def build_backbone(name, pretrained):
    if name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        backbone = models.resnet18(weights=weights)
        feature_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
    elif name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        backbone = models.efficientnet_b0(weights=weights)
        feature_dim = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
    elif name == "convnext_tiny":
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
        backbone = models.convnext_tiny(weights=weights)
        feature_dim = backbone.classifier[2].in_features
        backbone.classifier[2] = nn.Identity()
    else:
        raise ValueError(f"지원하지 않는 backbone: {name}")
    return backbone, int(feature_dim)

class TypeConditionalRegressor(nn.Module):
    def __init__(self, backbone_name, pretrained=True, dropout=0.15):
        super().__init__()
        self.backbone_name = backbone_name
        self.backbone, feature_dim = build_backbone(backbone_name, pretrained)
        self.type_embedding = nn.Embedding(len(TYPE_ORDER), 32)
        self.head = nn.Sequential(
            nn.Linear(feature_dim + 32, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )
        lows = [TYPE_RANGES[name][0] for name in TYPE_ORDER]
        highs = [TYPE_RANGES[name][1] for name in TYPE_ORDER]
        self.register_buffer("type_low", torch.tensor(lows, dtype=torch.float32))
        self.register_buffer("type_high", torch.tensor(highs, dtype=torch.float32))

    def forward(self, images, type_index):
        features = self.backbone(images)
        type_features = self.type_embedding(type_index)
        raw = self.head(torch.cat([features, type_features], dim=1)).squeeze(1)
        normalized = torch.sigmoid(raw)
        low = self.type_low[type_index]
        high = self.type_high[type_index]
        return low + normalized * (high - low)

def regression_loss(pred_m, target_m, type_index):
    lows = pred_m.new_tensor([TYPE_RANGES[name][0] for name in TYPE_ORDER])
    highs = pred_m.new_tensor([TYPE_RANGES[name][1] for name in TYPE_ORDER])
    low = lows[type_index]
    span = highs[type_index] - low
    pred_norm = (pred_m - low) / span
    target_norm = (target_m - low) / span
    robust = F.smooth_l1_loss(pred_norm, target_norm, beta=0.05)
    meter_term = torch.mean(((pred_m - target_m) / 20.0) ** 2)
    return robust + 0.20 * meter_term

def metric_payload(
    frame, truth_column="target_m", prediction_column="pred_m",
    require_all_types=True,
):
    required = {"type", truth_column, prediction_column}
    if required - set(frame.columns):
        raise AssertionError(f"metric 열 누락: {sorted(required - set(frame.columns))}")
    values = frame[[truth_column, prediction_column]].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise AssertionError("metric 입력에 non-finite 값이 있습니다")
    error = frame[prediction_column].to_numpy(float) - frame[truth_column].to_numpy(float)
    overall = float(np.sqrt(np.mean(error ** 2)))
    by_type = {}
    present_types = [
        type_name for type_name in TYPE_ORDER
        if frame["type"].eq(type_name).any()
    ]
    if require_all_types and present_types != TYPE_ORDER:
        missing_types = [name for name in TYPE_ORDER if name not in present_types]
        raise AssertionError(f"metric에 유형이 없습니다: {missing_types}")
    for type_name in present_types:
        part = frame.loc[frame["type"].eq(type_name)]
        part_error = part[prediction_column].to_numpy(float) - part[truth_column].to_numpy(float)
        by_type[type_name] = {
            "n": int(len(part)),
            "rmse_m": float(np.sqrt(np.mean(part_error ** 2))),
            "mae_m": float(np.mean(np.abs(part_error))),
            "bias_m": float(np.mean(part_error)),
        }
    weighted_mse = sum(
        item["n"] * item["rmse_m"] ** 2 for item in by_type.values()
    ) / sum(item["n"] for item in by_type.values())
    if not np.isclose(overall ** 2, weighted_mse, rtol=1e-10, atol=1e-12):
        raise AssertionError("전체 RMSE와 유형별 weighted MSE가 일치하지 않습니다")
    return {
        "overall_rmse_m": overall,
        "worst_type_rmse_m": max(item["rmse_m"] for item in by_type.values()),
        "by_type": by_type,
    }


In [ ]:
auth.authenticate_user()
STORAGE_CLIENT = storage.Client(project=PROJECT_ID)
BUCKET = STORAGE_CLIENT.bucket(BUCKET_NAME)
BUCKET.reload()
dataset_seal = download_json(BUCKET, DATASET_SEAL_OBJECT)
if dataset_seal["dataset_version"] != DATASET_VERSION:
    raise AssertionError("dataset version 불일치")
if dataset_seal["data_profile"] != DATA_PROFILE:
    raise AssertionError("dataset profile 불일치")
if not all(dataset_seal["assertions"].values()):
    raise AssertionError("upstream dataset seal assertion 실패")
if RUN_LEVEL != "smoke" and dataset_seal["counts"]["val"] != EXPECTED_FULL_VAL_N:
    raise AssertionError("pilot/full 학습은 전체 validation 30,808개가 필요합니다")
parent_selection = None
if RUN_MODE == "full_retrain":
    parent_selection = download_json(
        BUCKET,
        f"dcc/mission3/selection/{PILOT_SELECTION_ID}/model_selection_seal.json",
    )
    if parent_selection["status"] != "MODEL_SELECTION_FROZEN":
        raise AssertionError("pilot selection seal 상태 오류")
    if parent_selection.get("schema_version") != "dcc-m3-model-selection-seal-v3":
        raise AssertionError("pilot selection seal schema 오류")
    if parent_selection.get("selection_id") != PILOT_SELECTION_ID:
        raise AssertionError("pilot selection ID 오류")
    if parent_selection.get("selection_mode") != "pilot_select" or parent_selection.get("training_scope") != "pilot":
        raise AssertionError("pilot selection scope 오류")
    parent_assertions = parent_selection.get("assertions", {})
    if not parent_assertions or not all(parent_assertions.values()):
        raise AssertionError("pilot selection assertion 실패")
    if parent_selection.get("winner_model_id") != parent_selection["winner"].get("model_id"):
        raise AssertionError("pilot selection winner ID 오류")
    pilot_architecture_payload = parent_selection["winner"].get("architecture_payload")
    if (
        pilot_architecture_payload is not None
        and canonical_sha256(pilot_architecture_payload)
        != parent_selection["winner"].get("architecture_fingerprint")
    ):
        raise AssertionError("pilot winner architecture fingerprint 오류")
    frozen_winner = parent_selection["winner"]
    if frozen_winner["kind"] == "full_frame":
        frozen_backbone = frozen_winner["backbone"]
    elif frozen_winner["kind"] == "yolo_roi":
        frozen_backbone = frozen_winner["fallback_backbone"]
    elif frozen_winner["kind"] == "type_router":
        components = frozen_winner.get("components", {})
        full_component = components.get("full_frame")
        if not full_component or full_component.get("kind") != "full_frame":
            raise AssertionError("router full-frame component 계약 오류")
        if full_component.get("model_id") != frozen_winner["component_model_ids"]["full_frame"]:
            raise AssertionError("router full-frame component ID 불일치")
        frozen_backbone = full_component["backbone"]
    else:
        raise RuntimeError(f"지원하지 않는 pilot winner kind: {frozen_winner['kind']}")
    CANDIDATES = [str(frozen_backbone)]
    if dataset_seal["data_profile"] != "full" or dataset_seal["counts"]["train"] < 200_000:
        raise AssertionError("full 재학습용 전체 train dataset이 아닙니다")
    print("frozen pilot winner:", parent_selection["winner_model_id"])
    print("full-frame architecture to retrain:", CANDIDATES[0])
print("train rows:", dataset_seal["counts"]["train"])
print("val rows:", dataset_seal["counts"]["val"])


frozen pilot winner: type-router-convnext-tiny-yolo26s-roi-v1
full-frame architecture to retrain: convnext_tiny
train rows: 246308
val rows: 30808


## 1. GCS shard를 Colab SSD에 한 번만 내려받기

학습 batch마다 GCS object를 다시 열지 않습니다. descriptor의 generation, byte 수, SHA-256을
검증한 로컬 tar만 WebDataset으로 읽습니다.


In [ ]:
def materialize_split(split_name):
    descriptors = dataset_seal["shards"][split_name]
    required = sum(int(item["bytes"]) for item in descriptors)
    if shutil.disk_usage("/content").free < required + 8 * 1024**3:
        raise RuntimeError(f"{split_name} shard를 받을 로컬 디스크가 부족합니다")
    paths = []
    for index, descriptor in enumerate(descriptors, start=1):
        _, object_name = parse_gs_uri(descriptor["uri"])
        local = (
            LOCAL_SHARDS / split_name / descriptor["archive_key"]
            / Path(object_name).name
        )
        if local.exists() and (
            local.stat().st_size != int(descriptor["bytes"])
            or sha256_file(local) != descriptor["sha256"]
        ):
            raise RuntimeError(f"기존 local shard 충돌: {local}")
        if not local.exists():
            download_object(
                BUCKET, object_name, local, generation=descriptor["generation"]
            )
        if local.stat().st_size != int(descriptor["bytes"]):
            raise AssertionError("downloaded shard size 불일치")
        if sha256_file(local) != descriptor["sha256"]:
            raise AssertionError("downloaded shard SHA 불일치")
        paths.append(str(local))
        if index % 10 == 0 or index == len(descriptors):
            print(split_name, index, "/", len(descriptors))
    return paths

TRAIN_SHARDS = materialize_split("train")
VAL_SHARDS = materialize_split("val")
print("local train shards:", len(TRAIN_SHARDS))
print("local val shards:", len(VAL_SHARDS))


train 10 / 147
train 20 / 147
train 30 / 147
train 40 / 147
train 50 / 147
train 60 / 147
train 70 / 147
train 80 / 147
train 90 / 147
train 100 / 147
train 110 / 147
train 120 / 147
train 130 / 147
train 140 / 147
train 147 / 147
val 10 / 30
val 20 / 30
val 30 / 30
local train shards: 147
local val shards: 30


## 2. 누수 방지 loader와 모델 정의

`stem`은 결과 연결용 ID로만 보존됩니다. 모델 `forward()`에는 image와 type index만 전달합니다.
좌우·상하 반전, random crop, mosaic은 controlled baseline에서 사용하지 않습니다.


In [ ]:
train_transform = build_transform(IMAGE_SIZE, training=True, augment=AUGMENT)
eval_transform = build_transform(IMAGE_SIZE, training=False, augment=False)

def ensure_meta(value):
    if isinstance(value, (bytes, bytearray)):
        return json.loads(value.decode("utf-8"))
    return value

def sample_mapper(sample, transform):
    image, metadata = sample
    metadata = ensure_meta(metadata)
    required = {"stem", "type", "target_m"}
    if required - set(metadata):
        raise AssertionError(f"shard metadata 누락: {required - set(metadata)}")
    type_name = str(metadata["type"])
    if type_name not in TYPE_TO_INDEX:
        raise AssertionError(f"예상하지 않은 type: {type_name}")
    return (
        transform(image),
        torch.tensor(TYPE_TO_INDEX[type_name], dtype=torch.long),
        torch.tensor(float(metadata["target_m"]), dtype=torch.float32),
        str(metadata["stem"]),
        type_name,
    )

def make_loader(split_name, training):
    urls = TRAIN_SHARDS if split_name == "train" else VAL_SHARDS
    transform = train_transform if training else eval_transform
    shard_shuffle = max(1, len(urls)) if training else False
    dataset = wds.WebDataset(
        urls,
        shardshuffle=shard_shuffle,
        empty_check=True,
        handler=wds.handlers.reraise_exception,
    )
    if training:
        dataset = dataset.shuffle(2_000, initial=500, rng=random.Random(SEED))
    dataset = (
        dataset
        .decode("pil")
        .to_tuple("jpg", "json")
        .map(lambda sample: sample_mapper(sample, transform))
        # worker별 마지막 작은 batch도 학습해 eligible train 행을 버리지 않습니다.
        .batched(BATCH_SIZE, partial=True)
    )
    return wds.WebLoader(
        dataset,
        batch_size=None,
        num_workers=NUM_WORKERS,
        persistent_workers=NUM_WORKERS > 0,
        pin_memory=DEVICE.type == "cuda",
    )

smoke_batch = next(iter(make_loader("train", training=True)))
images, type_indices, targets, stems, types = smoke_batch
assert images.ndim == 4 and images.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)
assert len(stems) == len(targets)
assert set(types).issubset(set(TYPE_ORDER))
print("loader smoke:", images.shape, targets.min().item(), targets.max().item())


loader smoke: torch.Size([64, 3, 224, 224]) 17.200000762939453 18.799999237060547


In [ ]:
RUN_CONFIG = {
    "schema_version": "dcc-m3-fullframe-config-v1",
    "run_id": RUN_ID,
    "run_level": RUN_LEVEL,
    "run_mode": RUN_MODE,
    "training_scope": "full" if RUN_MODE == "full_retrain" else "pilot",
    "parent_selection_id": PILOT_SELECTION_ID if RUN_MODE == "full_retrain" else None,
    "dataset_version": DATASET_VERSION,
    "dataset_config_sha256": dataset_seal["config_sha256"],
    "validation_stem_sha256": dataset_seal["validation_stem_sha256"],
    "candidates": CANDIDATES,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "augment": AUGMENT,
    "seed": SEED,
    "max_train_batches": MAX_TRAIN_BATCHES,
    "max_val_batches": MAX_VAL_BATCHES,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
}
RUN_CONFIG_SHA256 = canonical_sha256(RUN_CONFIG)
print("run config sha256:", RUN_CONFIG_SHA256)

def list_epoch_blobs(candidate):
    prefix = f"{RUN_PREFIX}/{candidate}/checkpoints/epoch-"
    rows = []
    for blob in STORAGE_CLIENT.list_blobs(BUCKET_NAME, prefix=prefix):
        name = Path(blob.name).stem
        try:
            epoch = int(name.split("-")[-1])
        except ValueError:
            continue
        rows.append((epoch, blob))
    return sorted(rows, key=lambda item: item[0])

def load_resume(candidate, model, optimizer, scheduler, scaler):
    rows = list_epoch_blobs(candidate)
    if not rows:
        return 0, float("inf"), [], 0.0
    epoch, blob = rows[-1]
    expected_prefix = f"{RUN_PREFIX}/{candidate}/checkpoints/epoch-"
    blob.reload()
    if not blob.name.startswith(expected_prefix):
        raise AssertionError("resume checkpoint prefix 불일치")
    if (blob.metadata or {}).get("artifact_type") != "m3_regressor_checkpoint":
        raise AssertionError("resume checkpoint artifact type 불일치")
    local = LOCAL_RUN / candidate / f"resume-epoch-{epoch:03d}.pt"
    download_object(BUCKET, blob.name, local, generation=blob.generation)
    checkpoint = torch.load(local, map_location="cpu", weights_only=False)
    if checkpoint.get("schema_version") != "dcc-m3-regressor-checkpoint-v1":
        raise AssertionError("resume checkpoint schema 불일치")
    if checkpoint["run_config_sha256"] != RUN_CONFIG_SHA256:
        raise RuntimeError(f"{candidate} resume config 충돌")
    if checkpoint.get("candidate") != candidate:
        raise AssertionError("resume checkpoint candidate 불일치")
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    scaler.load_state_dict(checkpoint["scaler"])
    rng_state = checkpoint.get("rng_state")
    if rng_state:
        random.setstate(rng_state["python"])
        np.random.set_state(rng_state["numpy"])
        torch.set_rng_state(rng_state["torch"])
        if torch.cuda.is_available():
            torch.cuda.set_rng_state_all(rng_state["cuda"])
    print(candidate, "resume epoch", epoch)
    return (
        int(epoch), float(checkpoint["best_rmse"]),
        list(checkpoint["history"]), float(checkpoint["elapsed_seconds"]),
    )

def evaluate_model(model, max_batches=None):
    model.eval()
    records = []
    with torch.inference_mode():
        for batch_index, batch in enumerate(make_loader("val", training=False)):
            if max_batches is not None and batch_index >= max_batches:
                break
            images, type_indices, targets, stems, type_names = batch
            images = images.to(DEVICE, non_blocking=True)
            type_indices = type_indices.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=DEVICE.type == "cuda"):
                predictions = model(images, type_indices)
            for stem, type_name, target, prediction in zip(
                stems, type_names, targets.numpy(), predictions.float().cpu().numpy()
            ):
                records.append({
                    "stem": str(stem), "type": str(type_name),
                    "target_m": float(target), "pred_m": float(prediction),
                })
    frame = pd.DataFrame(records).sort_values("stem", kind="mergesort").reset_index(drop=True)
    assert frame["stem"].is_unique
    return frame, metric_payload(
        frame, require_all_types=(RUN_LEVEL != "smoke")
    )


run config sha256: 8e1d1c0b1188f8edaab8ad0e0bd33ebdb9576eb7738d6067d84bbb973a47e136


## 3. 세 후보 순차 학습

epoch마다 GCS에 create-only checkpoint를 저장합니다. 같은 `RUN_ID`로 재실행하면 가장 높은
epoch를 검증한 뒤 이어서 학습합니다. config가 달라지면 새 `RUN_ID`가 필요합니다.


In [ ]:
candidate_results = []
for candidate in CANDIDATES:
    print("=" * 80)
    print("candidate:", candidate)
    candidate_dir = LOCAL_RUN / candidate
    candidate_dir.mkdir(parents=True, exist_ok=True)
    model = TypeConditionalRegressor(candidate, pretrained=True).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS))
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
    start_epoch, best_rmse, history, prior_elapsed = load_resume(
        candidate, model, optimizer, scheduler, scaler
    )
    run_started = time.perf_counter()
    best_epoch = (
        int(min(history, key=lambda row: row["val_rmse_m"])["epoch"])
        if history else None
    )

    for epoch in range(start_epoch + 1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        seen = 0
        epoch_started = time.perf_counter()
        for batch_index, batch in enumerate(make_loader("train", training=True)):
            if MAX_TRAIN_BATCHES is not None and batch_index >= MAX_TRAIN_BATCHES:
                break
            images, type_indices, targets, _, _ = batch
            images = images.to(DEVICE, non_blocking=True)
            type_indices = type_indices.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=DEVICE.type == "cuda"):
                predictions = model(images, type_indices)
                loss = regression_loss(predictions, targets, type_indices)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += float(loss.detach()) * len(targets)
            seen += len(targets)
            if batch_index == 199:
                elapsed = time.perf_counter() - epoch_started
                print(
                    f"{candidate} measured throughput: {seen/elapsed:.1f} images/s"
                )
        scheduler.step()
        val_frame, val_metrics = evaluate_model(model, MAX_VAL_BATCHES)
        current_rmse = val_metrics["overall_rmse_m"]
        is_best = current_rmse < best_rmse
        if is_best:
            best_rmse = current_rmse
            best_epoch = epoch
        total_elapsed = prior_elapsed + (time.perf_counter() - run_started)
        history.append({
            "epoch": epoch,
            "train_loss": running_loss / max(1, seen),
            "val_rmse_m": current_rmse,
            "val_worst_type_rmse_m": val_metrics["worst_type_rmse_m"],
            "seconds": time.perf_counter() - epoch_started,
            "is_best": is_best,
        })
        checkpoint = {
            "schema_version": "dcc-m3-regressor-checkpoint-v1",
            "run_config": RUN_CONFIG,
            "run_config_sha256": RUN_CONFIG_SHA256,
            "candidate": candidate,
            "epoch": epoch,
            "best_rmse": best_rmse,
            "history": history,
            "elapsed_seconds": total_elapsed,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "rng_state": {
                "python": random.getstate(),
                "numpy": np.random.get_state(),
                "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            },
        }
        checkpoint_path = candidate_dir / f"epoch-{epoch:03d}.pt"
        torch.save(checkpoint, checkpoint_path)
        upload_create_only(
            BUCKET,
            checkpoint_path,
            f"{RUN_PREFIX}/{candidate}/checkpoints/epoch-{epoch:03d}.pt",
            "m3_regressor_checkpoint",
        )
        print(candidate, history[-1])

    if best_epoch is None:
        raise RuntimeError(f"{candidate}: best epoch가 없습니다")
    best_blob_name = f"{RUN_PREFIX}/{candidate}/checkpoints/epoch-{best_epoch:03d}.pt"
    best_blob = BUCKET.blob(best_blob_name)
    best_blob.reload()
    if (best_blob.metadata or {}).get("artifact_type") != "m3_regressor_checkpoint":
        raise AssertionError("best checkpoint artifact type 불일치")
    best_descriptor = {
        "uri": f"gs://{BUCKET_NAME}/{best_blob.name}",
        "generation": int(best_blob.generation),
        "bytes": int(best_blob.size),
        "sha256": (best_blob.metadata or {})["sha256"],
        "artifact_type": "m3_regressor_checkpoint",
    }
    local_best = candidate_dir / "best.pt"
    download_object(
        BUCKET, best_blob.name, local_best, generation=best_blob.generation
    )
    best_checkpoint = torch.load(local_best, map_location="cpu", weights_only=False)
    if best_checkpoint.get("schema_version") != "dcc-m3-regressor-checkpoint-v1":
        raise AssertionError("best checkpoint schema 불일치")
    if best_checkpoint.get("run_config_sha256") != RUN_CONFIG_SHA256:
        raise AssertionError("best checkpoint run config 불일치")
    if best_checkpoint.get("candidate") != candidate:
        raise AssertionError("best checkpoint candidate 불일치")
    model.load_state_dict(best_checkpoint["model"])
    model.to(DEVICE)
    prediction_frame, metrics = evaluate_model(model, MAX_VAL_BATCHES)

    if RUN_LEVEL != "smoke":
        assert len(prediction_frame) == EXPECTED_FULL_VAL_N
        observed_val_sha = canonical_sha256(sorted(prediction_frame["stem"].tolist()))
        assert observed_val_sha == dataset_seal["validation_stem_sha256"]
    else:
        observed_val_sha = canonical_sha256(sorted(prediction_frame["stem"].tolist()))
    observed_target_sha = canonical_target_sha256(prediction_frame, "target_m")
    if RUN_LEVEL != "smoke":
        assert observed_target_sha == dataset_seal["validation_target_sha256"]

    prediction_path = candidate_dir / "validation_predictions.parquet"
    prediction_frame.to_parquet(prediction_path, index=False, compression="zstd")
    prediction_descriptor = upload_create_only(
        BUCKET,
        prediction_path,
        f"{RUN_PREFIX}/{candidate}/validation_predictions.parquet",
        "m3_validation_predictions",
    )
    candidate_results.append({
        "model_id": f"fullframe-{candidate}",
        "kind": "full_frame",
        "backbone": candidate,
        "image_size": IMAGE_SIZE,
        "architecture_fingerprint": canonical_sha256({
            "kind": "full_frame",
            "backbone": candidate,
            "image_size": IMAGE_SIZE,
            "type_conditioning": "32d_embedding",
            "output": "sigmoid_type_physical_range_meter",
        }),
        "metrics": metrics,
        "validation_rows": int(len(prediction_frame)),
        "validation_stem_sha256": observed_val_sha,
        "validation_target_sha256": observed_target_sha,
        "checkpoint": best_descriptor,
        "prediction": prediction_descriptor,
        "best_epoch": int(best_epoch),
        "history": history,
    })
    del model, optimizer, scheduler, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


candidate: convnext_tiny
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 150MB/s]
/tmp/ipykernel_7313/1980782579.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")


convnext_tiny resume epoch 3


/tmp/ipykernel_7313/1980782579.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=DEVICE.type == "cuda"):


convnext_tiny measured throughput: 39.4 images/s


/tmp/ipykernel_7313/4205785147.py:86: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=DEVICE.type == "cuda"):


... (2 outputs left out)

convnext_tiny measured throughput: 39.2 images/s


/tmp/ipykernel_7313/4205785147.py:86: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=DEVICE.type == "cuda"):


convnext_tiny {'epoch': 5, 'train_loss': 0.0014978145364874455, 'val_rmse_m': 0.26450450986932456, 'val_worst_type_rmse_m': 0.7134307556171529, 'seconds': 6746.58405637, 'is_best': True}


/tmp/ipykernel_7313/4205785147.py:86: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=DEVICE.type == "cuda"):


## 4. validation 결과 봉인

이 셀은 후보별 prediction·checkpoint hash와 전체·8유형 RMSE를 묶습니다. smoke bundle은
`eligible_for_selection=False`이므로 선택 노트북에서 거부됩니다.


In [ ]:
if RUN_MODE == "pilot_compare" and len(candidate_results) < 3:
    raise AssertionError("Mission 3-1 pilot 비교에는 후보 3개 이상이 필요합니다")
if RUN_MODE == "full_retrain" and len(candidate_results) != 1:
    raise AssertionError("full 재학습은 pilot에서 동결한 architecture 1개만 허용합니다")
val_hashes = {item["validation_stem_sha256"] for item in candidate_results}
val_target_hashes = {item["validation_target_sha256"] for item in candidate_results}
val_counts = {item["validation_rows"] for item in candidate_results}
assert len(val_hashes) == 1 and len(val_target_hashes) == 1 and len(val_counts) == 1

leaderboard = pd.DataFrame([
    {
        "model_id": item["model_id"],
        "overall_rmse_m": item["metrics"]["overall_rmse_m"],
        "worst_type_rmse_m": item["metrics"]["worst_type_rmse_m"],
        "best_epoch": item["best_epoch"],
    }
    for item in candidate_results
]).sort_values(["overall_rmse_m", "worst_type_rmse_m"], kind="mergesort")
display(leaderboard)

package_versions = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "webdataset": getattr(wds, "__version__", "unknown"),
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
}
bundle = {
    "status": "CANDIDATES_COMPLETE",
    "schema_version": "dcc-m3-candidate-bundle-v1",
    "run_id": RUN_ID,
    "run_level": RUN_LEVEL,
    "eligible_for_selection": RUN_MODE in {"pilot_compare", "full_retrain"},
    "training_scope": "full" if RUN_MODE == "full_retrain" else "pilot",
    "parent_selection_id": PILOT_SELECTION_ID if RUN_MODE == "full_retrain" else None,
    "dataset_version": DATASET_VERSION,
    "dataset_config_sha256": dataset_seal["config_sha256"],
    "validation_rows": int(next(iter(val_counts))),
    "validation_stem_sha256": next(iter(val_hashes)),
    "validation_target_sha256": next(iter(val_target_hashes)),
    "candidate_count": len(candidate_results),
    "candidates": candidate_results,
    "preliminary_winner": leaderboard.iloc[0]["model_id"],
    "selection_rule": [
        "minimum validation overall meter RMSE",
        "minimum worst-type meter RMSE",
        "reproducibility and resource use if statistically tied",
    ],
    "run_config": RUN_CONFIG,
    "run_config_sha256": RUN_CONFIG_SHA256,
    "package_versions": package_versions,
}
bundle_descriptor = upload_json_create_only(
    BUCKET,
    bundle,
    f"{RUN_PREFIX}/candidate_bundle.json",
    "m3_candidate_bundle",
)
print("FULL-FRAME CANDIDATES COMPLETE")
print("bundle:", bundle_descriptor["uri"])


                  model_id  overall_rmse_m  worst_type_rmse_m  best_epoch
0  fullframe-convnext_tiny        0.264505           0.713431           5

FULL-FRAME CANDIDATES COMPLETE
bundle: gs://YOUR_GCS_BUCKET/dcc/mission3/runs/m3-fullframe-router-full-v2/candidate_bundle.json


## 다음 단계

pilot에서는 03C의 YOLO+ROI 후보를 이어서 실행한 뒤 03D에서 type-router까지 비교합니다.
`full_retrain`에서는 이 full-frame bundle이 03C의 고정 fallback과 router component가 되므로
03C `full_retrain`을 다음 순서로 실행합니다.
